## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.special import zeta 

## STAGE 1 -- Physical constants (SI units)

In [20]:
# These are the *exact* defining values of the SI, fixed by the 2019
# redefinition of the SI base units (BIPM / CODATA). Before 2019 these
# were measured quantities with small uncertainties; since 2019 they
# are exact by definition, and everything else (e.g. the kilogram) is
# defined in terms of them.

h   = 6.62607015e-34    # Planck constant,        J s
c   = 2.99792458e8      # speed of light in vacuum, m/s
k_B = 1.380649e-23      # Boltzmann constant,     J/K

# The "second radiation constant" c2 = hc/k_B (units: m K) is the
# single combination of constants that sets the scale of the Planck
# spectrum. It appears throughout as the coefficient in the
# dimensionless variable x = hc/(lambda k_B T) = c2/(lambda T).

c2 = h * c / k_B  # CODATA value: 1.4387768775e-2 m K
# print(c2)

## STAGE 2 -- Blackbody temperature

In [23]:
T = 5000.0   # kelvin (a mid-range star) 

# ===================================================================
# Planck's law -- both forms (Stage 4 will use the wavelength form)
# ===================================================================

def planck_B_lambda(wavelength_m, temperature_K):
    """Spectral radiance B_lambda(lambda, T), units W / (m^2 sr m).
    Power per unit area, per unit solid angle, per unit WAVELENGTH
    interval. See Marr & Wilkin (2012), Eq. (1); NIST/NBS TN 910-8,
    Ch. 12. 
    """
    #I've asked for citations  that's why It's citing the source also, can remove later if not wanted 
    # also if grammarly makes any suggestions please dont correct people
    
    x = c2 / (wavelength_m * temperature_K)
   
    # np.expm1(x) = exp(x) - 1, computed accurately even when x is small 
    # direct exp(x)-1 loses precision by catastrophic cancellation for small x.
    
    return (2.0 * h * c**2 / wavelength_m**5) / np.expm1(x)


def planck_B_nu(freq_Hz, temperature_K):
    
    """Spectral radiance B_nu(nu, T), units W / (m^2 sr Hz).
    Power per unit area, per unit solid angle, per unit FREQUENCY
    interval. NOT obtained from B_lambda by substituting nu = c/lambda
    -- see the Jacobian discussion in the write-up.
    """
    x = h * freq_Hz / (k_B * temperature_K)
    return (2.0 * h * freq_Hz**3 / c**2) / np.expm1(x)


# ===================================================================
# The photon-number spectral shape, in the dimensionless variable
#   x = hc / (lambda k_B T)
# This shape, g(x) = x^2 / (e^x - 1), is temperature-independent --
# see the derivation in the write-up (Sections 4-5). We use it to
# decide the numerical wavelength window before ever choosing a T.
# ===================================================================
def g_shape(x):
    return x**2 / np.expm1(x)


def find_x_window(tolerance, x_scan_lo=1e-8, x_scan_hi=80.0, n_scan=600_000):
    
    """Return (x_lo, x_hi) such that the probability outside
    [x_lo, x_hi] is approximately `tolerance` (split evenly between
    the two tails). Uses a fine log-spaced scan + cumulative
    trapezoidal integration, exactly the same technique Stage 7 will
    use to build the sampling CDF.
    """
    x_scan = np.logspace(np.log10(x_scan_lo), np.log10(x_scan_hi), n_scan)
    g_vals = g_shape(x_scan)
    cdf = np.concatenate(
        [[0.0], np.cumsum(0.5 * (g_vals[1:] + g_vals[:-1]) * np.diff(x_scan))]
    )
    cdf /= cdf[-1]
    x_lo = np.interp(tolerance / 2.0, cdf, x_scan)
    x_hi = np.interp(1.0 - tolerance / 2.0, cdf, x_scan)
    return x_lo, x_hi


# Implementation choice: we use tolerance = 1e-6 rather than 1e-8.
# The reason is explained in the write-up: the long-wavelength
# (Rayleigh-Jeans) side of the photon-number spectrum falls off only
# as a POWER law (g(x) ~ x as x -> 0), not exponentially, so pushing
# the tolerance from 1e-6 to 1e-8 buys very little extra fidelity but
# multiplies the required wavelength range (and hence the dynamic
# range the grid must resolve) by roughly a factor of 10. 1e-6 is
# still an extremely tight criterion for a simulation.

TOLERANCE = 1e-6

## STAGE 3 -- Build the wavelength grid

In [26]:
def build_wavelength_grid(temperature_K, tolerance=TOLERANCE, n_grid=200_000):
    """Build a wavelength grid that is uniform in log(x), where
    x = hc/(lambda k_B T). A grid that is uniform in *wavelength*
    would either waste millions of points on the essentially-empty
    far tail, or under-resolve the peak -- see the write-up for a
    worked demonstration. A log-spaced grid in x automatically
    concentrates points where the spectrum actually varies.
    """
    x_lo, x_hi = find_x_window(tolerance)
    x_grid = np.logspace(np.log10(x_lo), np.log10(x_hi), n_grid)
    wavelength_grid = c2 / (temperature_K * x_grid)  # descending in wavelength
    wavelength_grid = wavelength_grid[::-1]           # make it ascending
    return wavelength_grid, x_lo, x_hi

## STAGE 5 -- Photon-number spectral distribution (numbers to be changed later)

In [29]:
def photon_number_spectrum(wavelength_m, temperature_K):
    """Shape of dN/dlambda (photon-number spectral density), up to an
    overall constant that will cancel once we normalize it into a
    probability density. Derivation: dN/dlambda = B_lambda / (hc/lambda).
    """
    x = c2 / (wavelength_m * temperature_K)
    return (1.0 / wavelength_m**4) / np.expm1(x)


def photon_number_spectrum_nu(freq_Hz, temperature_K):
    """Shape of dN/dnu (photon-number spectral density in frequency).
    Derivation: dN/dnu = B_nu / (h nu).
    """
    x = h * freq_Hz / (k_B * temperature_K)
    return freq_Hz**2 / np.expm1(x)

## STAGES 6-7 -- Normalize and build the CDF

In [34]:
def build_pdf_and_cdf(wavelength_grid, temperature_K):
    pdf_unnormalized = photon_number_spectrum(wavelength_grid, temperature_K)
    normalization = np.trapz(pdf_unnormalized, wavelength_grid)
    pdf = pdf_unnormalized / normalization

    # Cumulative trapezoidal integration: cdf[i] = integral of pdf
    # from wavelength_grid[0] to wavelength_grid[i].
    increments = 0.5 * (pdf[1:] + pdf[:-1]) * np.diff(wavelength_grid)
    cdf = np.concatenate([[0.0], np.cumsum(increments)])
    cdf /= cdf[-1]   # guard against tiny round-off so cdf[-1] == 1 exactly
    return pdf, cdf

## STAGES 8-9 -- Draw random numbers and invert the CDF

In [37]:
def sample_photon_wavelengths(n_photons, wavelength_grid, cdf, rng):
    """Inverse-transform sampling: draw u ~ Uniform(0,1), then find the
    wavelength lambda such that CDF(lambda) = u, by linear
    interpolation on the numerical CDF table.
    """
    u = rng.uniform(0.0, 1.0, n_photons)
    return np.interp(u, cdf, wavelength_grid)

## STAGE 12 -- Statistics and validation

In [ ]:
def summarize(samples, wavelength_grid, pdf, temperature_K, label=""):
    photon_energies = h * c / samples          # E = hc/lambda, joules
    mean_energy = photon_energies.mean()
    mean_energy_theory = (np.pi**4 / (30.0 * zeta(3))) * k_B * temperature_K

    mean_wavelength = samples.mean()
    i_peak = np.argmax(pdf)
    mode_wavelength = wavelength_grid[i_peak]

    x_peak_photon_nu = brentq(lambda x: 2 * (1 - np.exp(-x)) - x, 0.3, 5.0)
    mode_frequency = x_peak_photon_nu * k_B * temperature_K / h

    print(f"--- {label} (T = {temperature_K:.0f} K, N = {len(samples):,}) ---")
    print(f"  mean photon energy   : {mean_energy:.4e} J "
          f"({mean_energy/(k_B*temperature_K):.4f} k_B T; theory 2.7012 k_B T)")
    print(f"  mean wavelength      : {mean_wavelength*1e9:.2f} nm")
    print(f"  most probable lambda : {mode_wavelength*1e9:.2f} nm")
    print(f"  most probable nu     : {mode_frequency:.4e} Hz "
          f"(<-> lambda = {c/mode_frequency*1e9:.2f} nm, NOT equal to most probable lambda)")
    return {
        "mean_energy": mean_energy,
        "mean_energy_theory": mean_energy_theory,
        "mean_wavelength": mean_wavelength,
        "mode_wavelength": mode_wavelength,
        "mode_frequency": mode_frequency,
    }


def fraction_in_range(samples, lam_min, lam_max):
    inside = (samples >= lam_min) & (samples <= lam_max)
    return inside.mean()

## STAGES 10-11 -- Plots

In [41]:
PLOT_XMAX_MULT = 6.0   # display window: a few times the peak wavelength,
                        # NOT the full numerical sampling domain (which
                        # extends much further to satisfy the tolerance
                        # criterion but contains negligible probability)


def plot_theoretical_distribution(wavelength_grid, pdf, temperature_K, outpath):
    lam_peak = wavelength_grid[np.argmax(pdf)]
    xmax = PLOT_XMAX_MULT * lam_peak
    mask = wavelength_grid <= xmax

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(wavelength_grid[mask] * 1e9, pdf[mask] * 1e-9, color="#1f5fa8", lw=2)
    ax.axvline(lam_peak * 1e9, color="gray", ls="--", lw=1,
               label=f"peak = {lam_peak*1e9:.0f} nm")
    ax.set_xlabel("wavelength  (nm)")
    ax.set_ylabel(r"photon-number probability density  $p(\lambda)$  (nm$^{-1}$)")
    ax.set_title(f"Theoretical blackbody photon-number distribution, T = {temperature_K:.0f} K")
    ax.legend()
    fig.tight_layout()
    fig.savefig(outpath, dpi=150)
    plt.close(fig)


def plot_histogram_vs_theory(samples, wavelength_grid, pdf, temperature_K, outpath, n_bins=120):
    lam_peak = wavelength_grid[np.argmax(pdf)]
    xmax = PLOT_XMAX_MULT * lam_peak

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(samples * 1e9, bins=n_bins, range=(0, xmax * 1e9), density=True,
            color="#a8c8e8", edgecolor="none", label=f"Monte Carlo, N={len(samples):,}")
    mask = wavelength_grid <= xmax
    ax.plot(wavelength_grid[mask] * 1e9, pdf[mask] * 1e-9, color="#c0392b", lw=2,
            label="theoretical p(\u03bb)")
    ax.set_xlabel("wavelength  (nm)")
    ax.set_ylabel(r"probability density  (nm$^{-1}$)")
    ax.set_title(f"Simulated photon histogram vs. theory, T = {temperature_K:.0f} K")
    ax.legend()
    fig.tight_layout()
    fig.savefig(outpath, dpi=150)
    plt.close(fig)


def plot_N_convergence(wavelength_grid, cdf, pdf, temperature_K, N_list, outpath):
    lam_peak = wavelength_grid[np.argmax(pdf)]
    xmax = PLOT_XMAX_MULT * lam_peak
    mask = wavelength_grid <= xmax

    fig, axes = plt.subplots(1, len(N_list), figsize=(4 * len(N_list), 4), sharey=True)
    for ax, N in zip(axes, N_list):
        rng_local = np.random.default_rng(42)   # same seed each time for a fair comparison
        samples = sample_photon_wavelengths(N, wavelength_grid, cdf, rng_local)
        ax.hist(samples * 1e9, bins=80, range=(0, xmax * 1e9), density=True,
                color="#a8c8e8", edgecolor="none")
        ax.plot(wavelength_grid[mask] * 1e9, pdf[mask] * 1e-9, color="#c0392b", lw=1.5)
        ax.set_title(f"N = {N:,}")
        ax.set_xlabel("wavelength (nm)")
    axes[0].set_ylabel(r"probability density  (nm$^{-1}$)")
    fig.suptitle(f"Convergence of the Monte Carlo histogram as N increases (T = {temperature_K:.0f} K)")
    fig.tight_layout()
    fig.savefig(outpath, dpi=150)
    plt.close(fig)


def plot_temperature_comparison(T_list, outpath):
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    colors = ["#2c3e91", "#1f8a70", "#e07b0e", "#c0392b"]
    for temperature_K, color in zip(T_list, colors):
        wl_grid, _, _ = build_wavelength_grid(temperature_K)
        pdf_T, _ = build_pdf_and_cdf(wl_grid, temperature_K)
        lam_peak = wl_grid[np.argmax(pdf_T)]
        mask = wl_grid <= 5000e-9
        ax.plot(wl_grid[mask] * 1e9, pdf_T[mask] * 1e-9, color=color, lw=2,
                label=f"T = {temperature_K:.0f} K (peak {lam_peak*1e9:.0f} nm)")
    ax.set_xlabel("wavelength  (nm)")
    ax.set_ylabel(r"photon-number probability density  (nm$^{-1}$)")
    ax.set_title("Photon-number distribution shifts with temperature")
    ax.legend()
    fig.tight_layout()
    fig.savefig(outpath, dpi=150)
    plt.close(fig)


In [11]:
# ===================================================================
# Run the core simulation at the default temperature, plus all
# validation checks and plots.
# ===================================================================
if __name__ == "__main__":
    rng = np.random.default_rng(42)   # fixed seed -> reproducible results

    wavelength_grid, x_lo, x_hi = build_wavelength_grid(T)
    pdf, cdf = build_pdf_and_cdf(wavelength_grid, T)

    N_MAIN = 100_000
    samples_main = sample_photon_wavelengths(N_MAIN, wavelength_grid, cdf, rng)
    stats_main = summarize(samples_main, wavelength_grid, pdf, T, label="main run")

    frac = fraction_in_range(samples_main, 400e-9, 700e-9)
    print(f"  fraction of photons in visible range (400-700 nm): {frac:.4f}")

    # Wien's displacement law check, using the ENERGY spectral
    # radiance B_lambda directly (NOT the photon-number distribution
    # -- this is a deliberately separate calculation).
    x_wien_energy_lambda = brentq(lambda x: 5 * (1 - np.exp(-x)) - x, 0.5, 6.0)
    lambda_peak_energy = c2 / (T * x_wien_energy_lambda)
    b_codata = 2.897771955e-3
    print(f"\nWien check (energy B_lambda peak): "
          f"lambda_peak * T = {lambda_peak_energy*T:.6e} m K "
          f"(CODATA b = {b_codata:.6e} m K)")

    print(f"\nNumerical-window check: x in [{x_lo:.5g}, {x_hi:.5g}], "
          f"i.e. lambda in [{wavelength_grid[0]*1e9:.2f} nm, "
          f"{wavelength_grid[-1]*1e6:.3f} um] for T = {T:.0f} K")

    # -----------------------------------------------------------
    # Plots
    # -----------------------------------------------------------
    plot_theoretical_distribution(wavelength_grid, pdf, T, "plot1_theoretical_distribution.png")
    plot_histogram_vs_theory(samples_main, wavelength_grid, pdf, T, "plot2_histogram_vs_theory.png")
    plot_N_convergence(wavelength_grid, cdf, pdf, T, [1_000, 10_000, 100_000, 1_000_000],
                        "plot3_N_convergence.png")
    plot_temperature_comparison([1000, 3000, 5000, 6000], "plot4_temperature_comparison.png")
    print("\nSaved plot1..plot4 PNG files.")

    # -----------------------------------------------------------
    # Temperature-dependence stats table (Check 4)
    # -----------------------------------------------------------
    print("\n--- Temperature dependence ---")
    for temperature_K in [1000, 3000, 5000, 6000]:
        wl_grid_T, _, _ = build_wavelength_grid(temperature_K)
        pdf_T, cdf_T = build_pdf_and_cdf(wl_grid_T, temperature_K)
        rng_T = np.random.default_rng(42)
        samples_T = sample_photon_wavelengths(50_000, wl_grid_T, cdf_T, rng_T)
        summarize(samples_T, wl_grid_T, pdf_T, temperature_K, label=f"T={temperature_K}")

--- main run (T = 5000 K, N = 100,000) ---
  mean photon energy   : 1.8633e-19 J (2.6992 k_B T; theory 2.7012 k_B T)
  mean wavelength      : 1967.49 nm
  most probable lambda : 733.95 nm
  most probable nu     : 1.6603e+14 Hz (<-> lambda = 1805.67 nm, NOT equal to most probable lambda)
  fraction of photons in visible range (400-700 nm): 0.1643

Wien check (energy B_lambda peak): lambda_peak * T = 2.897772e-03 m K (CODATA b = 2.897772e-03 m K)

Numerical-window check: x in [0.0015509, 19.693], i.e. lambda in [146.12 nm, 1855.383 um] for T = 5000 K

Saved plot1..plot4 PNG files.

--- Temperature dependence ---
--- T=1000 (T = 1000 K, N = 50,000) ---
  mean photon energy   : 3.7258e-20 J (2.6986 k_B T; theory 2.7012 k_B T)
  mean wavelength      : 9894.82 nm
  most probable lambda : 3669.73 nm
  most probable nu     : 3.3206e+13 Hz (<-> lambda = 9028.33 nm, NOT equal to most probable lambda)
--- T=3000 (T = 3000 K, N = 50,000) ---
  mean photon energy   : 1.1177e-19 J (2.6986 k_B T; the